# parsing.ms_office.markitdown.utils

> Shared MarkItDown conversion and embedded-image utilities for Linux and Windows.

In [ ]:
# |default_exp parsing.ms_office.markitdown.utils

In [ ]:
# | hide
from nbdev.showdoc import *

## Shared configuration

In [ ]:
# | export
import base64
import binascii
import os
import re
import subprocess
import sys
import warnings
from io import BytesIO
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory
from zipfile import BadZipFile, ZipFile

from dotenv import load_dotenv
from lxml import etree
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
# | export
OFFICE_EXTENSIONS = frozenset({
    ".doc", ".docx", ".odt",
    ".ppt", ".pptx", ".odp",
    ".xls", ".xlsx", ".xlsm", ".xlsb", ".ods",
    ".csv", ".tsv",
})

IMAGE_EXTENSION_BY_MIME = {
    "jpeg": ".jpg",
    "jpg": ".jpg",
    "png": ".png",
    "gif": ".gif",
    "webp": ".webp",
    "bmp": ".bmp",
    "tiff": ".tiff",
    "svg+xml": ".svg",
    "wmf": ".wmf",
    "x-wmf": ".wmf",
    "emf": ".emf",
    "x-emf": ".emf",
    "vnd.microsoft.icon": ".ico",
}

MARKDOWN_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[(?P<alt>[^\]]*)\]\(\s*)data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>\s*\))",
    flags=re.IGNORECASE,
)
HTML_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix><img\b[^>]*?\bsrc\s*=\s*(?P<quote>[\"']))data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>(?P=quote)[^>]*>)",
    flags=re.IGNORECASE,
)
DATA_IMAGE_RE = MARKDOWN_DATA_IMAGE_RE


def _find_project_root(start: Path | str | None = None) -> Path:
    """Find PROJ_ROOT from the environment or a parent pyproject.toml."""
    configured_root = os.getenv("PROJ_ROOT")
    if configured_root:
        project_root = Path(configured_root).expanduser().resolve()
        if not project_root.is_dir():
            raise FileNotFoundError(f"PROJ_ROOT is not a directory: {project_root}")
        return project_root

    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not find pyproject.toml above {current}")


PROJ_ROOT = _find_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Project environment file not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)


def get_office_files_root() -> Path:
    """Read and validate OFFICE_FILES_ROOT from PROJ_ROOT/.env."""
    configured_root = os.getenv("OFFICE_FILES_ROOT")
    if not configured_root:
        raise RuntimeError(f"OFFICE_FILES_ROOT is not configured in {ENV_FILE}")

    root = Path(configured_root).expanduser()
    if not root.is_absolute():
        root = PROJ_ROOT / root
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"OFFICE_FILES_ROOT is not a directory: {root}")
    return root


## Office conversion

In [ ]:
# | export
_PPTX_NAMESPACES = {
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "p": "http://schemas.openxmlformats.org/presentationml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}
_PPTX_EMBED_ATTRIBUTE = f"{{{_PPTX_NAMESPACES['r']}}}embed"


def _remove_unusable_pptx_pictures(slide_xml: bytes) -> tuple[bytes, int]:
    """Remove picture shapes that have no embedded image relationship."""
    document = etree.fromstring(slide_xml)
    removed = 0
    for picture in document.xpath(".//p:pic", namespaces=_PPTX_NAMESPACES):
        blip = picture.find(f".//{{{_PPTX_NAMESPACES['a']}}}blip")
        if blip is not None and blip.get(_PPTX_EMBED_ATTRIBUTE):
            continue
        parent = picture.getparent()
        if parent is not None:
            parent.remove(picture)
            removed += 1

    if removed == 0:
        return slide_xml, 0
    return (
        etree.tostring(
            document,
            encoding="UTF-8",
            xml_declaration=True,
            standalone=True,
        ),
        removed,
    )


def _normalize_mpo_as_jpeg(image_data: bytes) -> tuple[bytes, bool]:
    """Keep the first MPO frame and remove its MPF index without recompression."""
    if b"MPF\x00" not in image_data:
        return image_data, False

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", Image.DecompressionBombWarning)
        with Image.open(BytesIO(image_data)) as image:
            if image.format != "MPO":
                return image_data, False
            frame_end = len(image_data)
            if image.n_frames > 1:
                image.seek(1)
                frame_end = image.offset

    first_frame = image_data[:frame_end]
    if not first_frame.startswith(b"\xff\xd8"):
        raise ValueError("MPO first frame is not a JPEG stream")

    normalized = bytearray(first_frame[:2])
    position = 2
    removed_mpf = False
    while position < len(first_frame):
        marker_start = position
        if first_frame[position] != 0xFF:
            raise ValueError(f"Invalid JPEG marker at byte {position}")
        while position < len(first_frame) and first_frame[position] == 0xFF:
            position += 1
        if position >= len(first_frame):
            raise ValueError("Truncated JPEG marker in MPO first frame")

        marker = first_frame[position]
        position += 1
        if marker in {0xD8, 0xD9, 0x01} or 0xD0 <= marker <= 0xD7:
            normalized.extend(first_frame[marker_start:position])
            if marker == 0xD9:
                break
            continue
        if position + 2 > len(first_frame):
            raise ValueError("Truncated JPEG segment length in MPO first frame")

        segment_length = int.from_bytes(first_frame[position : position + 2], "big")
        segment_end = position + segment_length
        if segment_length < 2 or segment_end > len(first_frame):
            raise ValueError("Invalid JPEG segment length in MPO first frame")
        if marker == 0xDA:
            normalized.extend(first_frame[marker_start:])
            break

        payload = first_frame[position + 2 : segment_end]
        if marker == 0xE2 and payload.startswith(b"MPF\x00"):
            removed_mpf = True
        else:
            normalized.extend(first_frame[marker_start:segment_end])
        position = segment_end

    if not removed_mpf:
        raise ValueError("MPO image has no removable MPF segment")
    return bytes(normalized), True


def _sanitize_pptx_for_markitdown(source: Path, target: Path) -> dict[str, int]:
    """Write a temporary PPTX without image constructs MarkItDown cannot read."""
    repairs = {"removed_pictures": 0, "normalized_mpo_images": 0}
    with ZipFile(source, "r") as input_archive, ZipFile(target, "w") as output_archive:
        output_archive.comment = input_archive.comment
        for archive_entry in input_archive.infolist():
            payload = input_archive.read(archive_entry)
            archive_path = archive_entry.filename.lower()
            if (
                archive_path.startswith("ppt/slides/slide")
                and archive_path.endswith(".xml")
            ):
                payload, removed = _remove_unusable_pptx_pictures(payload)
                repairs["removed_pictures"] += removed
            elif archive_path.startswith("ppt/media/") and archive_path.endswith(
                (".jpeg", ".jpg", ".jpe")
            ):
                payload, normalized = _normalize_mpo_as_jpeg(payload)
                repairs["normalized_mpo_images"] += int(normalized)
            output_archive.writestr(archive_entry, payload)
    return repairs


def _markitdown_command(source: Path, target: Path) -> list[str]:
    return [
        sys.executable,
        "-m",
        "markitdown",
        str(source),
        "-o",
        str(target),
        "--keep-data-uris",
    ]


def _run_markitdown(source: Path, target: Path) -> None:
    subprocess.run(
        _markitdown_command(source, target),
        check=True,
        capture_output=True,
        text=True,
    )


def _conversion_error_message(error: Exception) -> str:
    if isinstance(error, subprocess.CalledProcessError) and error.stderr:
        stderr_lines = [line.strip() for line in error.stderr.splitlines() if line.strip()]
        if stderr_lines:
            return stderr_lines[-1]
    return str(error)


def convert_office_to_md(
    root_folder: Path | str,
    output_root: Path | str | None = None,
    *,
    overwrite: bool = False,
    show_progress: bool = True,
) -> dict[str, object]:
    """Recursively convert supported Office files to mirrored Markdown output."""
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Office root is not a directory: {root}")

    md_root = Path(output_root).expanduser() if output_root else root / ".md"
    if not md_root.is_absolute():
        md_root = root / md_root
    md_root = md_root.resolve()
    md_root.mkdir(parents=True, exist_ok=True)

    source_files = sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in OFFICE_EXTENSIONS
        and not path.name.startswith("~$")
    )
    report: dict[str, object] = {
        "root": root,
        "output_root": md_root,
        "discovered": source_files,
        "converted": [],
        "skipped": [],
        "failed": [],
    }

    for source in tqdm(
        source_files,
        desc="Converting Office files",
        unit="file",
        dynamic_ncols=True,
        disable=not show_progress,
    ):
        relative_source = source.relative_to(root)
        markdown_file = (
            md_root / relative_source.parent / source.stem / f"{source.stem}.md"
        )
        if markdown_file.exists() and not overwrite:
            report["skipped"].append(markdown_file)
            tqdm.write(f"Skipped existing: {markdown_file}")
            continue

        markdown_file.parent.mkdir(parents=True, exist_ok=True)
        conversion_error: Exception | None = None
        try:
            _run_markitdown(source, markdown_file)
        except OSError as error:
            conversion_error = error
        except subprocess.CalledProcessError as error:
            conversion_error = error
            if source.suffix.lower() == ".pptx":
                try:
                    with TemporaryDirectory(prefix="ribosome-pptx-") as temp_dir:
                        sanitized_source = Path(temp_dir) / source.name
                        repairs = _sanitize_pptx_for_markitdown(
                            source,
                            sanitized_source,
                        )
                        if any(repairs.values()):
                            tqdm.write(
                                f"Retrying sanitized PPTX: {source} "
                                f"({repairs['removed_pictures']} unusable picture(s) "
                                f"removed, {repairs['normalized_mpo_images']} MPO "
                                "image(s) normalized)"
                            )
                            _run_markitdown(sanitized_source, markdown_file)
                            conversion_error = None
                except (
                    BadZipFile,
                    Image.DecompressionBombError,
                    OSError,
                    ValueError,
                    etree.XMLSyntaxError,
                    subprocess.CalledProcessError,
                ) as retry_error:
                    conversion_error = retry_error

        if conversion_error is not None:
            report["failed"].append((source, conversion_error))
            tqdm.write(
                f"Failed: {source}: {_conversion_error_message(conversion_error)}"
            )
            continue

        report["converted"].append(markdown_file)
        tqdm.write(f"Converted: {source} -> {markdown_file}")

    return report


## Embedded-image extraction

In [ ]:
# | export
def _image_suffix(mime_subtype: str) -> str:
    """Return a safe filename extension for an image MIME subtype."""
    normalized = mime_subtype.lower()
    if normalized in IMAGE_EXTENSION_BY_MIME:
        return IMAGE_EXTENSION_BY_MIME[normalized]
    safe_subtype = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")
    return f".{safe_subtype or 'bin'}"


def extract_base64_images(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = "img",
) -> int:
    """Extract Markdown and HTML data-URI images into a relative folder."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        raise FileNotFoundError(f"Markdown file not found: {markdown_file}")

    relative_image_dir = Path(image_output_folder)
    if relative_image_dir.is_absolute() or ".." in relative_image_dir.parts:
        raise ValueError("image_output_folder must stay inside the Markdown directory")

    image_dir = markdown_file.parent / relative_image_dir
    markdown_image_dir = PurePosixPath(relative_image_dir.as_posix())
    content = markdown_file.read_text(encoding="utf-8")
    extracted_count = 0

    def write_image(match: re.Match, alt_text: str) -> str | None:
        nonlocal extracted_count
        encoded = "".join(match.group("data").split())
        encoded += "=" * (-len(encoded) % 4)
        try:
            image_data = base64.b64decode(encoded, validate=True)
        except (ValueError, binascii.Error) as error:
            print(f"Invalid base64 image in {markdown_file}: {error}")
            return None

        extracted_count += 1
        safe_alt = re.sub(r"[^\w.-]+", "_", alt_text, flags=re.UNICODE).strip("._")
        safe_alt = safe_alt[:50] or "image"
        image_name = (
            f"{extracted_count:04d}_{safe_alt}{_image_suffix(match.group('mime'))}"
        )
        image_dir.mkdir(parents=True, exist_ok=True)
        (image_dir / image_name).write_bytes(image_data)
        return (markdown_image_dir / image_name).as_posix()

    def replace_markdown_image(match: re.Match) -> str:
        image_link = write_image(match, match.group("alt"))
        if image_link is None:
            return match.group(0)
        return f'{match.group("prefix")}{image_link}{match.group("suffix").lstrip()}'

    def replace_html_image(match: re.Match) -> str:
        image_link = write_image(match, "image")
        if image_link is None:
            return match.group(0)
        return f'{match.group("prefix")}{image_link}{match.group("suffix")}'

    rewritten = MARKDOWN_DATA_IMAGE_RE.sub(replace_markdown_image, content)
    rewritten = HTML_DATA_IMAGE_RE.sub(replace_html_image, rewritten)
    if rewritten != content:
        markdown_file.write_text(rewritten, encoding="utf-8")
    return extracted_count


def extract_base64_from_md(
    root_folder: Path | str,
    image_output_folder: Path | str = "img",
    *,
    show_progress: bool = True,
) -> dict[str, object]:
    """Recursively extract data-URI images from Markdown files below root."""
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Markdown root is not a directory: {root}")

    report: dict[str, object] = {
        "processed": [],
        "failed": [],
        "image_count": 0,
        "images_extracted": 0,
    }
    markdown_files = sorted(
        path for path in root.rglob("*.md") if path.is_file()
    )
    for markdown_file in tqdm(
        markdown_files,
        desc="Extracting base64 images",
        unit="file",
        dynamic_ncols=True,
        disable=not show_progress,
    ):
        try:
            image_count = extract_base64_images(markdown_file, image_output_folder)
        except (OSError, ValueError) as error:
            report["failed"].append((markdown_file, error))
            tqdm.write(f"Failed to extract images from {markdown_file}: {error}")
            continue

        report["processed"].append(markdown_file)
        report["image_count"] += image_count
        report["images_extracted"] += image_count
        tqdm.write(f"Extracted {image_count} image(s): {markdown_file}")

    return report


## Complete workflow

In [ ]:
# | export
def process_office_files(
    root_folder: Path | str | None = None,
    *,
    output_root: Path | str | None = None,
    overwrite: bool = False,
    image_output_folder: Path | str = "img",
    show_progress: bool = True,
) -> dict[str, object]:
    """Convert Office files and extract their embedded data-URI images."""
    root = get_office_files_root() if root_folder is None else Path(root_folder)
    root = root.expanduser().resolve()
    conversion = convert_office_to_md(
        root,
        output_root,
        overwrite=overwrite,
        show_progress=show_progress,
    )
    extraction = extract_base64_from_md(
        conversion["output_root"],
        image_output_folder=image_output_folder,
        show_progress=show_progress,
    )

    tqdm.write(
        "Finished: "
        f"{len(conversion['converted'])} converted, "
        f"{len(conversion['skipped'])} skipped, "
        f"{len(conversion['failed'])} conversion failure(s), "
        f"{extraction['image_count']} image(s) extracted, "
        f"{len(extraction['failed'])} extraction failure(s)."
    )
    return {
        "root": root,
        "output_root": conversion["output_root"],
        "conversion": conversion,
        "extraction": extraction,
    }


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()